# Una solución empresarial completa

## Ahora llevaremos nuestro proyecto del día 1 al siguiente nivel

### DESAFÍO EMPRESARIAL:

Crear un producto que genere un folleto para una empresa que se utilizará para posibles clientes, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Consulte el final de este cuaderno para ver ejemplos de aplicaciones empresariales del mundo real.

Y recuerde: ¡siempre estoy disponible si tiene problemas o ideas! No dude en comunicarse conmigo.

In [1]:
# imports
# Si esto falla, verifica que esté ejecutándose desde un entorno "activado" con (llms) en el símbolo del sistema

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Inicialización y constantes and constants

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

La clave de API parece buena


In [3]:
# La clase para representar una Página Web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado, ahora con enlaces
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "Sin título"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Título de la Web:\n{self.title}\nContenido de la Web:\n{self.text}\n\n"

In [4]:
frog = Website("https://cursos.frogamesformacion.com")
print(frog.get_contents())
frog.links

Título de la Web:
Just a moment...
Contenido de la Web:
Enable JavaScript and cookies to continue




[]

## Primer paso: hacer que GPT-4o-mini determine qué enlaces son relevantes

### Usar una llamada a gpt-4o-mini para leer los enlaces en una página web y responder en JSON estructurado.
Debería decidir qué enlaces son relevantes y reemplazar los enlaces relativos como "/about" con "https://company.com/about".
Usaremos "one shot prompting" en las que proporcionamos un ejemplo de cómo debería responder en la solicitud.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. Imagínate intentar programar esto sin LLMs analizando la página web: ¡sería muy difícil!

Nota al margen: existe una técnica más avanzada llamada "Salidas estructuradas" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de inteligencia artificial Agentic.

In [6]:
link_system_prompt = "Se te proporciona una lista de enlaces que se encuentran en una página web. \
Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, \
como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.\n"
link_system_prompt += "Debes responder en JSON como en este ejemplo:"
link_system_prompt += """
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}
"""

In [7]:
print(link_system_prompt)

Se te proporciona una lista de enlaces que se encuentran en una página web. Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.
Debes responder en JSON como en este ejemplo:
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}



In [8]:
def get_links_user_prompt(website):
    user_prompt = f"Aquí hay una lista de enlaces de la página web {website.url} - "
    user_prompt += "Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. \
No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.\n"
    user_prompt += "Links (puede que algunos sean links relativos):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(frog))

Aquí hay una lista de enlaces de la página web https://cursos.frogamesformacion.com - Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.
Links (puede que algunos sean links relativos):



In [10]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [11]:
anthropic = Website("https://www.dronena.com")
anthropic.links

['http://windows.microsoft.com/es-ES/internet-explorer/downloads/ie',
 '#',
 '/Home/Default.aspx',
 'General/Nosotros/MisionVision.aspx',
 'General/FundacionNena/NuestraResponsabilidadSocial.aspx',
 'https://www.dronena.com/NuevaExperiencia/Principal/SerClienteNena',
 'General/SolicitudProveedor/SerProveedorNena.aspx',
 'General/Seguridad/Registro.aspx',
 'General/Contacto/Contacto.aspx',
 'http://blogdronena.wordpress.com/',
 'https://twitter.com/dronenave',
 'https://instagram.com/dronenave',
 'https://www.facebook.com/DronenaVe',
 'https://www.youtube.com/user/dronenave',
 'General/RSS/Rss.aspx',
 '#tab1',
 'General/Seguridad/ExpiroCuenta.aspx?height=400&width=655&modal=1',
 'https://gestor.dronena.com:8084/GESTORTOMAPEDIDO/CARRUSEL/06012025_085723.jpg',
 'https://gestor.dronena.com:8084/GESTORTOMAPEDIDO/CARRUSEL/08072025_094953.jpg',
 'https://gestor.dronena.com:8084/GESTORTOMAPEDIDO/CARRUSEL/02072025_034904.png',
 'https://gestor.dronena.com:8084/GESTORTOMAPEDIDO/CARRUSEL/06062025

In [12]:
get_links("https://www.dronena.com")

{'links': [{'type': 'Pagina Sobre nosotros',
   'url': 'https://www.dronena.com/General/Nosotros/MisionVision.aspx'},
  {'type': 'Pagina de la empresa', 'url': 'http://www.dronena.com/Home/'},
  {'type': 'Pagina de Proveedores',
   'url': 'https://www.dronena.com/General/SolicitudProveedor/SerProveedorNena.aspx'},
  {'type': 'Pagina de Responsabilidad Social',
   'url': 'https://www.dronena.com/General/FundacionNena/NuestraResponsabilidadSocial.aspx'},
  {'type': 'Contacto',
   'url': 'https://www.dronena.com/General/Contacto/Contacto.aspx'}]}

In [12]:
get_links("https://cursos.frogamesformacion.com")

{'links': [{'type': 'Pagina Sobre nosotros',
   'url': 'https://cursos.frogamesformacion.com'},
  {'type': 'Pagina de Instructores',
   'url': 'https://cursos.frogamesformacion.com/pages/instructores'},
  {'type': 'Pagina de Certificaciones',
   'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'},
  {'type': 'Pagina de Rutas',
   'url': 'https://cursos.frogamesformacion.com/pages/rutas'},
  {'type': 'Pagina de Premios',
   'url': 'https://cursos.frogamesformacion.com/pages/premios'},
  {'type': 'Pagina de Afiliados',
   'url': 'https://cursos.frogamesformacion.com/pages/afiliados'},
  {'type': 'Pagina de Cursos',
   'url': 'https://cursos.frogamesformacion.com/collections'},
  {'type': 'Cursos de Geometría Analítica',
   'url': 'https://cursos.frogamesformacion.com/courses/geometria-analitica-desde-cero'},
  {'type': 'Cursos de IA-productividad',
   'url': 'https://cursos.frogamesformacion.com/courses/ia-productividad'},
  {'type': 'Cursos de Python',
   'url': 'https:

## Segundo paso: ¡crea el folleto!

Reúne todos los detalles en otro mensaje para GPT4-o

In [13]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [14]:
print(get_all_details("https://www.dronena.com"))

Links encontrados: {'links': [{'type': 'Pagina Sobre nosotros', 'url': 'https://www.dronena.com/General/Nosotros/MisionVision.aspx'}, {'type': 'Responsabilidad Social', 'url': 'https://www.dronena.com/General/FundacionNena/NuestraResponsabilidadSocial.aspx'}, {'type': 'Página de Contacto', 'url': 'https://www.dronena.com/General/Contacto/Contacto.aspx'}]}
Landing page:
Título de la Web:

	Dronena

Contenido de la Web:
Debe Actualizar Internet Explorer a la versiÃ³n mas reciente para el correcto 
        funcionamiento de la PÃ¡gina Web.
Actualizar Ahora
Inicio
Nosotros
FundaciÃ³n Nena
Ser Cliente Nena
Ser Proveedor Nena
Suscribirme
ContÃ¡ctanos
Iniciar Sesion
Al registrarte, tendrÃ¡s acceso total a tu ficha de cliente y todos los servicios 
                            de Dronena.com
Si Tiene Activada la tecla Bloq MayÃºs es posible que escriba incorrectamente su contraseÃ±a.
Estamos Procesando su solicitud, espere
un momento por favor ...
Para navegar en nuestra pÃ¡gina web , recomenda

In [21]:
#system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa\
#y crea un folleto breve sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
#Incluye detalles sobre la cultura de la empresa, los clientes, las carreras/empleos y los cursos/packs para futuros empleos si tienes la información."

# O descomenta las líneas a continuación para obtener un folleto más humorístico: esto demuestra lo fácil que es incorporar el "tono":

system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa \
y crea un folleto breve, divertido y gracioso sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
Incluye detalles sobre la cultura de la empresa, los clientes y los cursos/packs para futuros empleos si tienes la información."


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [23]:
get_brochure_user_prompt("Dronena", "https://dronena.com")

Links encontrados: {'links': [{'type': 'Pagína Acerca de nosotros', 'url': 'https://www.dronena.com/Home/Default.aspx'}, {'type': 'Nuestra Responsabilidad Social', 'url': 'https://www.dronena.com/General/FundacionNena/NuestraResponsabilidadSocial.aspx'}, {'type': 'Ser Cliente Nena', 'url': 'https://www.dronena.com/NuevaExperiencia/Principal/SerClienteNena'}, {'type': 'Solicitud de Proveedor', 'url': 'https://www.dronena.com/General/SolicitudProveedor/SerProveedorNena.aspx'}]}


"Estás mirando una empresa llamada: Dronena\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\n\r\n\tDronena\r\n\nContenido de la Web:\nDebe Actualizar Internet Explorer a la versiÃ³n mas reciente para el correcto \r\n        funcionamiento de la PÃ¡gina Web.\nActualizar Ahora\nInicio\nNosotros\nFundaciÃ³n Nena\nSer Cliente Nena\nSer Proveedor Nena\nSuscribirme\nContÃ¡ctanos\nIniciar Sesion\nAl registrarte, tendrÃ¡s acceso total a tu ficha de cliente y todos los servicios \r\n                            de Dronena.com\nSi Tiene Activada la tecla Bloq MayÃºs es posible que escriba incorrectamente su contraseÃ±a.\nEstamos Procesando su solicitud, espere\nun momento por favor ...\nPara navegar en nuestra pÃ¡gina web , recomendamos usar Google Chrome\nDescÃ¡rguelo \r\n                aquÃ\xad\nInicio\nNoticias\nNena Mail\nEnlaces\nSEDE PRINCIPAL\nCa

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("Dronena", "https://www.dronena.com")

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://www.dronena.com/NuevaExperiencia/Principal/SerClienteNena'}, {'type': 'Nuestra Responsabilidad Social', 'url': 'https://www.dronena.com/General/Nosotros/MisionVision.aspx'}, {'type': 'Página de Contacto', 'url': 'https://www.dronena.com/General/Contacto/Contacto.aspx'}]}


# 🎉 ¡Bienvenido a Dronena! 🚀

¡Hola, futuro cliente, inversor o empleado! Prepara tu drone y tu mejor sonrisa, porque en Dronena estamos listos para despegar en una aventura emocionante. Aquí tienes todo lo que necesitas saber sobre nosotros (insisto, ¡todo!) en este divertido folleto. 

---

## ¿Quiénes Somos? 🤔
Somos Dronena, una empresa que no solo vuela alto, sino que también planea contribuir positivamente a la comunidad. Aunque estamos en Venezuela en Barquisimeto y Guarenas, ¡nuestros ideales vuelan incluso más lejos!

## La Cultura de Dronena 🌟
En Dronena, creemos que un ambiente de trabajo alegre es más productivo. Nos apasiona innovar y mejorar, y nos aseguramos de que cada empleado sienta que tiene un lugar en nuestra familia. Aquí no solo construimos drones, ¡también construimos relaciones! 

### Nuestros Valores:
- **Colaboración**: Porque dos (o más) cabezas piensan mejor que una, ¡sobre todo si tienen un drone junto a ellas!
- **Innovación**: ¡Siempre en la búsqueda de la próxima gran idea! 
- **Diversión**: ¿Quién dijo que trabajar no puede ser divertido? 

## Ser Cliente Nena 👩‍💻
Si decides unirte a nosotros, prepárate para un viaje sin obstáculos. Tú, sencillo y feliz, registrando tus datos. Desde Amazon hasta Zulia, ¡todos son bienvenidos! 🗺️ 

## Proveedores Bienvenidos 📦
En Dronena, buscamos aquellos que compartan nuestra pasión por la excelencia. ¿Tienes algo que ofrecer? Regístrate y formaremos un equipo increíble.

## Cursos y Packs Emocionantes 🧑‍🏫
¿Te gustaría trabajar en el mundo de los drones? ¡Estás de suerte! Ofrecemos cursos y packs que te prepararán para el futuro. Es como un vuelo de prueba hacia tu carrera soñada.

## Contáctanos 📞
¿Tienes dudas? ¿Un chiste sobre drones? ¡Nos encantaría escucharlo! Estamos aquí para ayudarte. 

Dirección:
- **Sede Principal**: Carrera 3 con Calle 3, Edif. Dronena, Zona Industrial III, Barquisimeto, Lara, Venezuela.
- **Sucursal Capital**: Zona Ind. del Este, Av. 2, Edif. Droguería Nena, Guarenas, Miranda, Venezuela.

*Nota:* Si intentas navegar nuestra web en Internet Explorer, ¡considera actualizar a Google Chrome! Puede que los drones no vuelen de la misma forma que tu navegador.

---

### 🚁 ¡Únete a la aventura Dronena!
Ya sea que busques ser cliente, proveedor o parte de nuestro extraordinario equipo, en Dronena hay un lugar para ti. ¡Hagamos que el futuro vuele alto juntos!

*No olvides sonreír cuando nos contacts, podrías estar a un paso más cerca de tu futuro brillante con Dronena.* 😄✨

---

*Disclaimer: Este folleto fue creado usando información limitada. Si encontraste un error, ¡no dudes en hacérnoslo saber (sin drones en mano, por favor!)!*

In [21]:
create_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Pagina de cursos', 'url': 'https://cursos.frogamesformacion.com/pages/rutas'}, {'type': 'Pagina de instructores', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Pagina de certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Nuestros clientes', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Frogames para empresas', 'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'}, {'type': 'Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Pagina principal', 'url': 'https://cursos.frogamesformacion.com'}, {'type': 'Afiliados', 'url': 'https://cursos.frogamesformacion.com/pages/afiliados'}]}


# Folleto de Frogames Formación

## 🚀 Acerca de Frogames Formación
Frogames Formación es una plataforma educativa galardonada, reconocida como la **"Enseñanza online de datos y competencias digitales más innovadora de Europa, 2023"**. Nos apasiona ayudar a las personas a convertirse en expertos en áreas como programación de videojuegos, inteligencia artificial, machine learning, desarrollo de aplicaciones y ciencia de datos. Con más de 500,000 estudiantes satisfechos, nuestra comunidad es un espacio vibrante donde se aprende mientras se divierte.

---

## 🎯 Nuestros Cursos y Rutas de Aprendizaje
Ofrecemos una amplia gama de **cursos online** que se adaptan a los diferentes niveles de habilidad de nuestros estudiantes:

### Rutas de Aprendizaje
- **Matemáticas desde Cero**
- **Desarrollo de Videojuegos**
- **Inteligencia Artificial**
- **Análisis de Datos**
- **Trading Algorítmico**
  
Cada ruta está diseñada para guiarte paso a paso, asegurando que adquieras las habilidades necesarias de forma progresiva. Los cursos ofrecen certificación digital verificada por **Blockchain**, lo que le añade valor a tu CV.

---

## 👨‍🏫 Instructores Expertos
Nuestro equipo de más de 20 instructores está compuesto por expertos en sus respectivas áreas. Entre ellos se destaca:
- **Juan Gabriel Gomila** - CEO y experto en educación.
- **María Santos** - Matemática y consultora educativa.
- **Ricardo Alberich** - Especialista en estadística.

---

## 🌐 Cultura de la Empresa
Frogames celebra un ambiente inclusivo y colaborativo, donde los estudiantes y empleados trabajan juntos para alcanzar el éxito. La filosofía de enseñanza se centra en la práctica, ofreciendo un aprendizaje que no solo es accesible, sino también divertido. Aquí, cada miembro de la comunidad comparte su pasión por aprender y crecer, apoyándose entre sí.

---

## 📈 Oportunidades de Carrera
Frogames no solo es un lugar para aprender, sino también para enseñarte a ti mismo y potencialmente formar parte del equipo. Ofrecemos oportunidades para convertirte en **afiliado** y recibir comisiones por cada venta que consigas. Buscamos personas apasionadas por la educación y la tecnología que deseen unirse a nuestro equipo.

---

## 🌟 Conclusión
Ya sea que estés buscando adquirir nuevas habilidades, una nueva carrera profesional o una oportunidad de inversión, Frogames Formación te ofrece las herramientas necesarias para avanzar en el mundo digital. **¡Únete a nosotros y comienza tu viaje de aprendizaje hoy!**

## Por último, una pequeña mejora

Con un pequeño ajuste, podemos cambiar esto para que los resultados se transmitan desde OpenAI,
con la animación de máquina de escribir habitual


In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://www.anthropic.com/company'}, {'type': 'Página de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Cursos', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Página de Noticias', 'url': 'https://www.anthropic.com/news'}]}


# Folleto de Anthropic

## Quiénes somos
Anthropic es una empresa de investigación y desarrollo de inteligencia artificial con sede en San Francisco, dedicada a construir sistemas de IA seguros, confiables e interpretables. Creemos en el potencial de la IA para impactar positivamente el mundo y estamos comprometidos a desarrollar tecnologías que beneficien a la humanidad.

## Nuestra misión
Nuestro objetivo es abordar los desafíos y oportunidades que presenta la IA, promoviendo una evolución responsable de esta tecnología. Nos enfocamos en la investigación y en la gobernanza, colaborando con diversas partes interesadas para garantizar que nuestros sistemas sean seguros y eficaces.

## Productos destacados
### Claude
- **Claude Opus 4.1**: Nuestro modelo de IA más inteligente y avanzado.
- **Claude Code**: Herramientas para crear aplicaciones impulsadas por IA.

## Compromisos de seguridad
Tratar la seguridad de la IA como una ciencia sistemática es esencial. Nuestros principios rectores y políticas, como la **Política de Escalado Responsable**, forman la base para un desarrollo seguro y fiable de la IA.

## Nuestra cultura
En Anthropic, valoramos:
- **Actuar por el bien global**: Tomamos decisiones que favorecen los resultados positivos para la humanidad en el largo plazo.
- **Cultura de apoyo**: Fomentamos un entorno de trabajo colaborativo, donde cada voz se escucha y se valora.
- **Compromiso con la transparencia**: Compartimos lo que aprendemos con la comunidad para que nuestra investigación sea accesible antes.

## Oportunidades de carrera
Estamos buscando personas creativas y apasionadas que deseen contribuir a un futuro seguro para la IA. Ofrecemos roles en diversas áreas, desde investigación y desarrollo hasta operaciones y política.

### Beneficios para empleados
- **Salud y bienestar**: Seguro médico integral, apoyo para fertilidad, y licencia parental de 22 semanas.
- **Apoyo financiero**: Salario competitivo, paquetes de acción y planes de ahorro para el retiro.
- **Estímulos educativos**: Protocolos de reembolso para formación continua y becas anuales.

## Anthropic Academy
Accede a la **Anthropic Academy**, donde puedes aprender a construir con Claude y mejorar tus habilidades en un ámbito en rápida evolución.

## Contáctanos
Si estás interesado en unirte a nuestro equipo o en saber más sobre nuestros productos y políticas, visita nuestra [página de carreras](#).

¡Únete a nosotros y se parte de la revolución de la IA segura!

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Links encontrados: {'links': [{'type': 'Pagina Sobre nosotros', 'url': 'https://huggingface.co/huggingface'}, {'type': 'Pagina de Carreras', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Pagina de Cursos', 'url': 'https://huggingface.co/learn'}, {'type': 'Pagina de Modelos', 'url': 'https://huggingface.co/models'}, {'type': 'Pagina de Espacios', 'url': 'https://huggingface.co/spaces'}, {'type': 'Pagina de Datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'Pagina de Documentación', 'url': 'https://huggingface.co/docs'}, {'type': 'Pagina de Precios', 'url': 'https://huggingface.co/pricing'}, {'type': 'Pagina de Blog', 'url': 'https://huggingface.co/blog'}, {'type': 'Pagina de Comunidad', 'url': 'https://discuss.huggingface.co'}]}


# Folleto de Hugging Face

## Acerca de la Empresa
**Hugging Face** es una comunidad de inteligencia artificial que trabaja para construir el futuro. La plataforma sirve como un punto de encuentro para que la comunidad de aprendizaje automático colabore en modelos, conjuntos de datos y aplicaciones. Con más de 1 millón de modelos y una creciente base de usuarios, Hugging Face se posiciona como un líder en la democratización del aprendizaje automático.

## Cultura de la Empresa
La cultura de Hugging Face se centra en la **colaboración**, la **transparencia** y el **apoyo comunitario**. La empresa está comprometida con la creación de herramientas de aprendizaje automático accesibles para todos, asegurando que todos los contribuyentes se sientan valorados e incluidos. La comunidad juega un papel vital; las contribuciones colectivas inspiran innovaciones y mejoran la plataforma.

## Clientes
Hugging Face trabaja con más de 50,000 organizaciones, incluyendo gigantes tecnológicos como [Meta](https://www.meta.com), [Microsoft](https://www.microsoft.com), y [Google](https://www.google.com). Estos clientes utilizan la plataforma para mejorar sus aplicaciones utilizando modelos de aprendizaje automático avanzados.

## Carreras y Empleos
Hugging Face está en constante búsqueda de talentos apasionados por la inteligencia artificial y el aprendizaje automático. Se ofrecen oportunidades en diversas áreas, desde investigación hasta desarrollo de software y soporte técnico. Los empleados tienen la oportunidad de trabajar en un entorno innovador y dinámico que fomenta el crecimiento profesional.

Puede consultar las posiciones abiertas actuales en la sección de [Carreras de Hugging Face](https://huggingface.co/jobs).

## Cursos y Packs para Futuras Carreras
Hugging Face ofrece una variedad de **cursos** diseñados para ayudar a los interesados a adquirir habilidades en el campo del aprendizaje automático. Algunos cursos destacados incluyen:

- **Curso de Modelos de Lenguaje Grandes (LLM)**: Aprende sobre modelos de lenguaje utilizando bibliotecas del ecosistema HF.
- **Curso de Agentes**: Construye y despliega tus propios agentes de inteligencia artificial.
- **Curso de Aprendizaje Reforzado Profundo (Deep RL)**: Introducción al aprendizaje reforzado con bibliotecas del ecosistema HF.
- **Curso de Visión por Computadora**: Aplica modelos de machine learning a problemas de visión.
- **Curso de Audio**: Aprende a utilizar transformadores en datos de audio.

Estos cursos están diseñados para proveer las habilidades necesarias que los futuros empleados necesitarán integralmente en el sector de la inteligencia artificial.

## Últimas Palabras
Hugging Face no solo es una plataforma para la comunidad de IA, sino un espacio vibrante y acogedor para todos los entusiastas del aprendizaje automático. Ya sea que busques colaborar, adquirir nuevas habilidades o unirte a un equipo innovador, Hugging Face tiene algo que ofrecerte. 

¡Únete a nosotros en esta emocionante travesía hacia el futuro de la inteligencia artificial!

---

Para más información, visita [Hugging Face](https://huggingface.co).

In [26]:
stream_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Página de Inicio', 'url': 'https://cursos.frogamesformacion.com'}, {'type': 'Página de Rutas', 'url': 'https://cursos.frogamesformacion.com/pages/rutas'}, {'type': 'Página de Instructores', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Página de Certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Página de Nuestros Clientes', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Página de Frogames para Empresas', 'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'}, {'type': 'Página de Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Página de Afiliados', 'url': 'https://cursos.frogamesformacion.com/pages/afiliados'}, {'type': 'Página de Cursos', 'url': 'https://cursos.frogamesformacion.com/collections'}]}


# Folleto de Frogames Formación

## Introducción
¡Bienvenido a Frogames! Somos una plataforma de aprendizaje en línea galardonada, reconocida como la **"Enseñanza online de datos y competencias digitales más innovadora de Europa, 2023"**. Nos dedicamos a ayudar a estudiantes de todas las edades a convertirse en expertos en diversas áreas como Programación de Videojuegos, Inteligencia Artificial, Machine Learning, Desarrollo de Apps y Data Science.

## Nuestro Compromiso
En Frogames, te ofrecemos:
- **Cursos en línea y formación de calidad** accesibles para toda la familia.
- **Rutas temáticas** que te guían paso a paso en tu aprendizaje.
- **Instructores expertos** que te acompañan durante todo el curso.
- **Certificados verificados por blockchain** que puedes compartir en tus redes sociales y utilizar para mejorar tu currículum.
- **Actualizaciones constantes** de contenido y rutas para asegurar que siempre estés aprendiendo lo más reciente.

## La Cultura de Frogames
Nuestra comunidad es el corazón de lo que hacemos. Creemos en:
- Fomentar un ambiente **divertido y motivador**, donde aprender es una experiencia agradable.
- Aprender en **una comunidad colaborativa**, donde estudiantes y profesores trabajan juntos.
- Proporcionar **oportunidades de formación** accesibles para todos, independientemente de su nivel de habilidad.

## Testimonios de Estudiantes
> "Me encanta aprender aquí. Es el lugar adecuado para aprender diferentes temáticas en un solo lugar. ¡Estoy encantado de estar junto a esta comunidad de amigos!" — **Eulogio Enamorado Pallares**

> "La calidad del equipo humano es la mejor que he visto en cursos online" — **Toni Peña Palencia**

## Oferta de Cursos y Rutas de Aprendizaje
Ofrecemos una variedad de **rutas de aprendizaje** diseñadas para diferentes niveles y áreas de interés:
- **Matemáticas desde Cero**
- **Desarrollo de Videojuegos**
- **Inteligencia Artificial**
- **Blockchain**
- **Machine Learning**
- **Ofimática y Productividad**

Cada ruta incluye acceso a múltiples cursos y materiales de capacitación.

## Carreras y Oportunidades de Empleo
Si estás interesado en unirte a nuestro equipo, Frogames también ofrece oportunidades para:
- **Convertirte en afiliado** y ganar una comisión por cada venta que consigas.
- Colaborar con nosotros como instructor, si tienes experiencia en el sector educativo.

## Formación para Empresas
Brindamos soluciones de formación a empresas que quieren que sus trabajadores desarrollen nuevas habilidades y mejoren su rendimiento en el trabajo. Con nuestros cursos, las empresas pueden mantener a su equipo actualizado en tecnologías emergentes.

## Únete a Nosotros
Estamos aquí para ayudarte a dar el siguiente paso en tu desarrollo profesional. ¡Empieza hoy mismo con un curso gratuito y forma parte de nuestra comunidad de más de **500,000 estudiantes satisfechos**!

### Contacto
Visita nuestro sitio web y ¡descubre todo lo que Frogames tiene para ofrecerte!

[¡Inicia tu aprendizaje hoy mismo en Frogames!](https://frogames.com)

---
**Frogames Formación**: Aprende, crece y disfruta mientras te conviertes en un experto en tecnología y más.

## Aplicaciones empresariales

En este ejercicio, ampliamos el código del día 1 para realizar múltiples llamadas a LLM y generar un documento.

En términos de técnicas, este es quizás el primer ejemplo de patrones de diseño de Agentic AI, ya que combinamos múltiples llamadas a LLM. Esto se abordará más en la semana 2 y luego volveremos a Agentic AI de manera importante en la semana 8, cuando construyamos una solución Agent completamente autónoma.

En términos de aplicaciones empresariales, generar contenido de esta manera es uno de los casos de uso más comunes. Al igual que con el resumen, esto se puede aplicar a cualquier vertical empresarial. Escriba contenido de marketing, genere un tutorial de producto a partir de una especificación, cree contenido de correo electrónico personalizado y mucho más. Explore cómo puede aplicar la generación de contenido a su negocio e intente crear un prototipo de prueba de concepto.